# MeetLens - Meeting & Lecture Summarisation (NLP Capstone, Project 18)
Team Techmorph | Run on **Google Colab** (Runtime > Change runtime type > T4 GPU).

Stages covered: Problem -> Dataset -> Cleaning -> Preprocessing -> EDA -> Features -> Models -> Evaluation (ROUGE).

**Problem:** convert long meeting/lecture conversations into short summaries.
**Models:** (A) TextRank baseline (TF-IDF), (B) pretrained DistilBART (no training), (C) **fine-tuned BART-base** on dialogue data.

In [ ]:
!pip -q install transformers datasets evaluate rouge_score accelerate scikit-learn matplotlib seaborn pandas

## 1. Dataset collection
SAMSum (dialogue -> summary) is used for training. If the name below fails on your Colab, search HuggingFace for 'samsum' and replace it.

In [ ]:
from datasets import load_dataset
ds = load_dataset("knkarthick/samsum")   # columns: id, dialogue, summary
print(ds)
print(ds['train'][0])

## 2. Data cleaning (missing values, duplicates, inconsistent rows)

In [ ]:
import pandas as pd
tr, va, te = [pd.DataFrame(ds[s]) for s in ['train','validation','test']]
for name, d in [('train',tr),('val',va),('test',te)]:
    print(name, d.shape, 'nulls:', d[['dialogue','summary']].isna().sum().sum(),
          'duplicates:', d.duplicated(subset=['dialogue']).sum())

def clean_df(d):
    d = d.dropna(subset=['dialogue','summary']).drop_duplicates(subset=['dialogue'])
    d = d[(d.dialogue.str.split().str.len() >= 10) & (d.summary.str.split().str.len() >= 3)]
    return d.reset_index(drop=True)
tr, va, te = clean_df(tr), clean_df(va), clean_df(te)
print('after cleaning:', len(tr), len(va), len(te))

## 3. Text preprocessing

In [ ]:
import re
def preprocess(t):
    t = re.sub(r'\[?\d{1,2}:\d{2}(?::\d{2})?\]?', ' ', t)           # timestamps
    t = re.sub(r'\b(um+|uh+|erm|hmm+|you know|i mean)\b,?', ' ', t, flags=re.I)  # fillers
    t = t.replace('\r\n', '\n')
    return re.sub(r'[ \t]+', ' ', t).strip()
for d in (tr, va, te):
    d['dialogue'] = d['dialogue'].map(preprocess)
print(tr.dialogue[0][:300])

## 4. EDA

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
tr['d_len'] = tr.dialogue.str.split().str.len()
tr['s_len'] = tr.summary.str.split().str.len()
tr['ratio'] = tr.s_len / tr.d_len
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(tr.d_len, bins=40, ax=ax[0]); ax[0].set_title('Dialogue length (words)')
sns.histplot(tr.s_len, bins=40, ax=ax[1]); ax[1].set_title('Summary length (words)')
sns.histplot(tr.ratio, bins=40, ax=ax[2]); ax[2].set_title('Summary / dialogue ratio')
plt.tight_layout(); plt.show()
print(tr[['d_len','s_len','ratio']].describe())

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as SW
words = [w for t in tr.summary for w in re.findall(r'[a-z]+', t.lower()) if w not in SW and len(w) > 2]
top = Counter(words).most_common(20)
plt.figure(figsize=(10,4)); sns.barplot(x=[w for w,_ in top], y=[c for _,c in top])
plt.xticks(rotation=60); plt.title('Top words in reference summaries'); plt.show()

## 5. Feature engineering + Model A: TextRank baseline (TF-IDF + cosine similarity + PageRank)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def split_sents(t):
    parts = re.split(r'(?<=[.!?])\s+|\n+', t)
    return [p.strip() for p in parts if len(p.split()) >= 3]

def textrank(text, k=3, d=0.85, iters=50):
    s = split_sents(text)
    if len(s) <= k: return ' '.join(s)
    try: X = TfidfVectorizer(stop_words='english').fit_transform(s)
    except ValueError: return ' '.join(s[:k])
    sim = cosine_similarity(X); np.fill_diagonal(sim, 0)
    rs = sim.sum(1, keepdims=True); rs[rs == 0] = 1
    M = sim / rs; n = len(s); r = np.ones(n) / n
    for _ in range(iters): r = (1-d)/n + d * M.T @ r
    idx = sorted(np.argsort(r)[::-1][:k])
    return ' '.join(s[i] for i in idx)

print(textrank(te.dialogue[0]))

## 6. Evaluation helper (ROUGE-1 / 2 / L)

In [ ]:
import evaluate
rouge = evaluate.load('rouge')
N = 200   # test samples for quick comparison; increase for final report
refs = te.summary[:N].tolist()
def score(preds, name):
    r = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    return {'Model': name, 'ROUGE-1': r['rouge1'], 'ROUGE-2': r['rouge2'], 'ROUGE-L': r['rougeL']}
results = []
results.append(score([textrank(t) for t in te.dialogue[:N]], 'TextRank (baseline)'))
pd.DataFrame(results)

## 7. Model B: pretrained DistilBART (no fine-tuning)

In [ ]:
import torch
from transformers import pipeline
dev = 0 if torch.cuda.is_available() else -1
pre = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6', device=dev)
preds = [o['summary_text'] for o in pre(te.dialogue[:N].tolist(), max_length=60, min_length=10,
                                         truncation=True, batch_size=16)]
results.append(score(preds, 'DistilBART (pretrained)'))
pd.DataFrame(results)

## 8. Model C: fine-tune BART-base on our data
This is the **model development** part. Takes ~15-25 min on a Colab T4 for 2 epochs.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
from datasets import Dataset
MODEL = 'facebook/bart-base'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

def tokenize(b):
    x = tok(b['dialogue'], max_length=512, truncation=True)
    x['labels'] = tok(text_target=b['summary'], max_length=96, truncation=True)['input_ids']
    return x
tr_ds = Dataset.from_pandas(tr[['dialogue','summary']]).map(tokenize, batched=True, remove_columns=['dialogue','summary'])
va_ds = Dataset.from_pandas(va[['dialogue','summary']]).map(tokenize, batched=True, remove_columns=['dialogue','summary'])

args = Seq2SeqTrainingArguments(
    output_dir='meetlens-bart', num_train_epochs=2, per_device_train_batch_size=8,
    per_device_eval_batch_size=16, learning_rate=5e-5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='no', fp16=torch.cuda.is_available(),
    logging_steps=100, report_to='none')
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
                         data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

In [ ]:
model.eval()
def generate(texts, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            g = model.generate(**enc, max_length=80, num_beams=4, early_stopping=True)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return out
preds = generate(te.dialogue[:N].tolist())
results.append(score(preds, 'BART-base (fine-tuned)'))
res = pd.DataFrame(results).round(4); res

## 9. Results comparison

In [ ]:
res.set_index('Model')[['ROUGE-1','ROUGE-2','ROUGE-L']].plot(kind='bar', figsize=(9,4))
plt.title('ROUGE comparison'); plt.xticks(rotation=15); plt.ylim(0, 0.7); plt.show()
for i in range(3):
    print('DIALOGUE :', te.dialogue[i][:250].replace('\n',' | '))
    print('REFERENCE:', te.summary[i]); print('MODEL    :', preds[i]); print('-'*60)

## 10. Save the model for the Streamlit app

In [ ]:
model.save_pretrained('meetlens_model'); tok.save_pretrained('meetlens_model')
!zip -rq meetlens_model.zip meetlens_model
from google.colab import files; files.download('meetlens_model.zip')

## Conclusion (fill after running)
- Best model by ROUGE-L: ...
- Observation: fine-tuned model vs baseline ...
- Limitations: trained on short chat dialogues (SAMSum); long meetings need chunking.
- Future scope: fine-tune on AMI/QMSum, add speech-to-text (Whisper), multilingual support.